In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv2D
import os
import math

# --- Parameter ---
DATA_FILE = '/kaggle/input/dataset-iconic-it/processed_masked_data.npz' # <-- SESUAIKAN JIKA PERLU
MODEL_SAVE_PATH = 'nusantara_reef_guardian_model.h5'
BATCH_SIZE = 1 # Batch size. Jaga agar tetap kecil (misal: 2 atau 4).
VALIDATION_SPLIT = 0.2 # 20% data untuk validasi

# --- Data Generator (Sequence) ---
# Kelas ini akan 'memberi makan' model batch per batch, menjaga RAM tetap rendah.
class CoralSequenceGenerator(tf.keras.utils.Sequence):
    def __init__(self, data_indices, x_data, y_data, batch_size):
        self.indices = data_indices
        self.x_data = x_data
        self.y_data = y_data
        self.batch_size = batch_size

    def __len__(self):
        # Jumlah batch per epoch
        return math.ceil(len(self.indices) / self.batch_size)

    def __getitem__(self, idx):
        # Mengambil satu batch indeks
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        
        # Mengambil data dari array di disk berdasarkan indeks
        batch_x = self.x_data[batch_indices]
        batch_y = self.y_data[batch_indices]
        return batch_x, batch_y

# --- Alur Utama ---
print(f"Membuka data dari {DATA_FILE} menggunakan memory mapping...")
if not os.path.exists(DATA_FILE):
    print(f"Error: File data '{DATA_FILE}' tidak ditemukan.")
    exit()

# mmap_mode='r' membuka file di disk tanpa memuat seluruhnya ke RAM
with np.load(DATA_FILE, mmap_mode='r') as data:
    x_data = data['X']
    y_data = data['y']
    
    print(f"Bentuk data input (X) di disk: {x_data.shape}")
    print(f"Bentuk data output (y) di disk: {y_data.shape}")

    # --- Membagi Indeks, Bukan Data ---
    num_samples = x_data.shape[0]
    indices = np.arange(num_samples)
    np.random.shuffle(indices) # Acak indeks

    split_point = int(num_samples * (1 - VALIDATION_SPLIT))
    train_indices = indices[:split_point]
    val_indices = indices[split_point:]
    
    print(f"Data dibagi menjadi {len(train_indices)} indeks pelatihan dan {len(val_indices)} indeks validasi.")

    # Membuat instance generator
    train_generator = CoralSequenceGenerator(train_indices, x_data, y_data, BATCH_SIZE)
    val_generator = CoralSequenceGenerator(val_indices, x_data, y_data, BATCH_SIZE)

    # --- Mendefinisikan Arsitektur Model ---
    print("\nMembangun arsitektur model ConvLSTM...")
    input_shape = x_data.shape[1:]
    model = Sequential([
        ConvLSTM2D(filters=32, kernel_size=(3, 3), padding='same', return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        ConvLSTM2D(filters=64, kernel_size=(3, 3), padding='same', return_sequences=False),
        BatchNormalization(),
        Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
    ])
    
    model.compile(optimizer='adam', loss='mse')
    model.summary()

    # --- Melatih Model Menggunakan Generator ---
    print("\nMemulai proses pelatihan model dengan Data Generator...")
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20
    )

    # --- Menyimpan Model ---
    print(f"\nPelatihan selesai. Menyimpan model ke {MODEL_SAVE_PATH}...")
    model.save(MODEL_SAVE_PATH)
    print("✅ Model berhasil disimpan!")